# Modern Embeddings Using all-MiniLM-L6-v2
## 1. Introduction
### 1.1 Imports and Loading Data

In [5]:
import json
import pandas as pd
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

input_path = '../data/processed/arxiv_text.parquet'

In [6]:
print(f"Loading data from {input_path}")
data = pd.read_parquet(input_path)

print(f"The shape of the input data is {data.shape}")
print(f"The columns or features are {data.columns.to_list()}")
print(f"First 5 rows:")
data.head()

Loading data from ../data/processed/arxiv_text.parquet
The shape of the input data is (169343, 7)
The columns or features are ['paper id', 'node idx', 'split', 'label', 'arxiv category', 'year', 'text']
First 5 rows:


,paper id,node idx,split,label,arxiv category,year,text
0,630234,104447,train,6,arxiv cs hc,2011,spreadsheets on the move an evaluation of mobi...
1,803423,15858,train,16,arxiv cs cv,2014,multi view metric learning for multi view vide...
2,1102481,107156,train,5,arxiv cs dc,2013,big data analytics in future internet of thing...
3,1532644,141536,train,24,arxiv cs lg,2014,machine learner for automated reasoning 0 4 an...
4,1810480,82077,train,4,arxiv cs cr,2011,cryptographic hardening of d sequences this pa...


### 1.2 Split The Data Into Test and Train

In [9]:
print("Split the data into training and test sets")

print("\nSplitting training set: no hyperparameter tuning this section, so combining validation and training sets")
X_train = data[data['split'] != 'test'].text
y_train = data[data['split'] != 'test'].label

print(f"Train shape: {X_train.shape}")
print(f"Train shape: {y_train.shape}")

print("\nSplitting test set")
X_test = data[data['split'] == 'test'].text
y_test = data[data['split'] == 'test'].label

print(f"Train shape: {X_test.shape}")
print(f"Train shape: {X_test.shape}")

Split the data into training and test sets

Splitting training set: no hyperparameter tuning this section, so combining validation and training sets
Train shape: (120740,)
Train shape: (120740,)

Splitting test set
Train shape: (48603,)
Train shape: (48603,)


## 2. Training Logistic Regression Using Sentence Embeddings (all-MiniLM-L6-v2)

In [4]:
# Load and encode
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
X_train_encoded = model.encode(X_train.tolist(), show_progress_bar=True)
X_test_encoded = model.encode(X_test.tolist(), show_progress_bar=True)

# Train and evaluate
lr_model = LogisticRegression(max_iter=10000)
lr_model.fit(X_train_encoded, y_train)
y_pred = lr_model.predict(X_test_encoded)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3774 [00:00<?, ?it/s]

Batches:   0%|          | 0/1519 [00:00<?, ?it/s]

Accuracy: 0.7249
F1-macro: 0.5351


## 3. Results

In [8]:
results = {
        "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
        "experiment_type": "Sentence Embeddings",
        "pretrained_model": "sentence-transformers/all-MiniLM-L6-v2",
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "train_shape": list(X_train_encoded.shape),
        "test_shape": list(X_test_encoded.shape)
    }
output_path = '../data/results/sentence/sentence_emb_results.json'

with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {output_path}")

Results saved to ../data/results/sentence/sentence_emb_results.json
